In [ ]:
import sys

from pathlib import Path


PROJECT_ROOT = Path.cwd()
SRC_ROOT = PROJECT_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))


from agentic_ai.proton import (
    open_proton_mail,
    close_proton_mail,
    ensure_logged_in,
    is_trash_page,
    open_trash,
    return_to_trash,
    open_trash_conversation,
    star_conversation,
)

from agentic_ai.agent import (
    create_email_triage_agent,
)

from agentic_ai.models import (
    EmailImportance,
)


In [ ]:

agents_file = PROJECT_ROOT / "src" / "agentic_ai" / "agents.py"

print("Path:", agents_file)
print("Exists:", agents_file.exists())

### Create the email agent

In [ ]:
email_agent = create_email_triage_agent()

In [ ]:
playwright, context, page = (
    await open_proton_mail(
        project_root=PROJECT_ROOT,
    )
)

await ensure_logged_in(page)


In [ ]:
await open_trash(page)

In [ ]:
print(
    "Trash:",
    await is_trash_page(page),
)

In [ ]:
rows = page.locator(
    "div.item-container--row"
)

first_row = rows.nth(0)

star_button = first_row.locator(
    "button.starbutton"
)

print("Star buttons:", await star_button.count())

if await star_button.count() == 1:
    print(
        await star_button.evaluate(
            """
            el => ({
                text: (el.innerText || "").trim(),
                ariaLabel: el.getAttribute("aria-label"),
                title: el.getAttribute("title"),
                className: el.className
            })
            """
        )
    )

In [ ]:
await star_conversation(
    page,
    index=0,
)

In [ ]:
await star_conversation(
    page,
    index=999999,
)

In [ ]:
email = await open_trash_conversation(
    page,
    index=0,
)

In [ ]:
await star_conversation(
    page,
    index=0,
)

In [ ]:
result = await email_agent.run(
    f"""
    Classify the importance of this email.

    Use these rules:

    "ignore":
    The email does not appear to require attention.

    "review":
    The email is potentially useful or relevant,
    but there is no clear urgent action.

    "urgent":
    The email requires timely attention,
    contains an important deadline,
    or could cause a significant consequence
    if ignored.

    Set action_required to true only when the email
    clearly requires the recipient to do something.

    Give one concise factual reason for the classification.

    Do not invent information that is not present.

    Email:

    {email.raw_text}
    """
)

In [ ]:
print(type(result.output))
print("Priority:", result.output.priority)
print( "Action required:", result.output.action_required,)
print("Reason:", result.output.reason)

In [ ]:
import sys

print(sys.executable)
print(sys.version)

````
conda activate agentic-ai   
```

```
conda deactivate
```